# CODI/KaVa controls and seeds — Kaggle worker

Run one experiment in parallel with the Colab worker. Use **Save Version → Save & Run All**, with GPU and Internet enabled. The notebook continues on Kaggle after the browser closes. Never assign the same experiment to Kaggle and Colab.

Recommended allocation: Kaggle runs `kava_random_seed0`, then both `codi_seed1` and `kava_seed1`. Colab runs the other controls and both seed-2 methods. Keeping each CODI/KaVa seed pair on one platform avoids a hardware confound.

In [ ]:
EXPERIMENT = "kava_random_seed0"
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with the same pinned commit used by Colab.
REPO_DIR = "/kaggle/working/latent-reasoning"
MAX_SECONDS = 39600  # 11h; internal guard stops 5% early.
EVAL_LIMIT = 200
RESUME_INPUT = ""  # Optional attached previous experiment directory.

KAGGLE_EXPERIMENTS = [
    "kava_random_seed0",
    "codi_seed1",
    "kava_seed1",
]
assert EXPERIMENT in KAGGLE_EXPERIMENTS


## 1. Clone the exact pinned revision

In [ ]:
import os, pathlib, shutil, subprocess, sys
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: public access")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("Pin RUN_COMMIT before the long run to:", commit)


## 2. Optional resume from a previous Kaggle version

If an earlier run exited with code 42, attach that notebook output as a Kaggle input and set `RESUME_INPUT` to its complete `controls_and_seeds/<experiment>` directory. Leave it empty for a fresh experiment.

In [ ]:
if RESUME_INPUT:
    source = pathlib.Path(RESUME_INPUT)
    assert source.name == EXPERIMENT, f"Resume source must end in {EXPERIMENT}"
    assert (source / "run_manifest.json").is_file(), "Resume manifest missing"
    assert list((source / "checkpoints").glob("step_*.pt")), "Resume checkpoint missing"
    target = pathlib.Path(REPO_DIR) / "outputs" / "controls_and_seeds" / EXPERIMENT
    if target.exists():
        shutil.rmtree(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, target)
    print("Restored:", target)
else:
    print("Fresh experiment; no checkpoint restore requested.")


## 3. Validate code, experiment contract, and GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
print("Torch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "scripts/validate_controls.py"], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)


## 4. Train/resume and evaluate

This is the long blocking cell. Exit 0 means the capped evaluation completed. Exit 42 means the checkpoint is safe but needs another Kaggle version.

In [ ]:
cmd = [
    sys.executable, "-u", "scripts/kaggle_control_runner.py",
    "--experiment", EXPERIMENT,
    "--output-root", REPO_DIR,
    "--max-seconds", str(MAX_SECONDS),
    "--eval-limit", str(EVAL_LIMIT),
    "--allow-environment-change",
]
print("Starting:", " ".join(cmd), flush=True)
result = subprocess.run(cmd, cwd=REPO_DIR)
print("Session exit code:", result.returncode)
if result.returncode not in (0, 42):
    raise RuntimeError(f"Experiment failed with exit code {result.returncode}")


## 5. Inspect and persist this Kaggle version

In [ ]:
status_path = pathlib.Path(REPO_DIR) / "status" / "controls_and_seeds" / f"{EXPERIMENT}.json"
print(status_path.read_text() if status_path.is_file() else "No status file")
output = pathlib.Path(REPO_DIR) / "outputs" / "controls_and_seeds" / EXPERIMENT
for path in sorted(output.rglob("*")):
    if path.is_file() and not path.name.endswith((".tmp", ".uploading")):
        print(f"{path.relative_to(output)}  {path.stat().st_size / 2**20:.1f} MiB")
print("Use Save Version with outputs enabled. Do not delete this version until it is imported to Drive.")


## Resume/import rule

For exit 42, save outputs, attach that version to a new Kaggle session, set `RESUME_INPUT`, and keep the same experiment. For exit 0, import the output into Drive using the Colab notebook's Kaggle-import cell before moving to the next Kaggle assignment.